In [1]:
import pandas as pd
from datasets import Dataset

/mnt/disk1/luanborges/verifiers_vlm/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('output.csv')
print(df.shape)
df.head()

(8016, 5)


,task,image,prompt,groundtruth,metadata
0,Subway Connections,images/img_0.png,How many single-color paths go from C to A? An...,1,"{""image_id"": ""b72dcafc-6225-4edb-ab9c-e59f57ea..."
1,Subway Connections,images/img_1.png,Count the one-colored routes that go from C to...,1,"{""image_id"": ""b72dcafc-6225-4edb-ab9c-e59f57ea..."
2,Subway Connections,images/img_2.png,How many single-color paths go from B to D? An...,1,"{""image_id"": ""b72dcafc-6225-4edb-ab9c-e59f57ea..."
3,Subway Connections,images/img_3.png,Count the one-colored routes that go from B to...,1,"{""image_id"": ""b72dcafc-6225-4edb-ab9c-e59f57ea..."
4,Subway Connections,images/img_4.png,How many single-color paths go from C to A? An...,1,"{""image_id"": ""34ea239e-0d35-406c-b04d-c6faa8ca..."


In [3]:
df = df[df['task'] == 'Line Plot Intersections']
df.head()

,task,image,prompt,groundtruth,metadata
960,Line Plot Intersections,images/img_960.png,How many times do the blue and red lines touch...,2,"{""gt"": 2, ""linewidth"": 2, ""left"": 2, ""resoluti..."
961,Line Plot Intersections,images/img_961.png,Count the intersection points where the blue a...,2,"{""gt"": 2, ""linewidth"": 2, ""left"": 2, ""resoluti..."
962,Line Plot Intersections,images/img_962.png,How many times do the blue and red lines touch...,2,"{""gt"": 2, ""linewidth"": 4, ""left"": 4, ""resoluti..."
963,Line Plot Intersections,images/img_963.png,Count the intersection points where the blue a...,2,"{""gt"": 2, ""linewidth"": 4, ""left"": 4, ""resoluti..."
964,Line Plot Intersections,images/img_964.png,How many times do the blue and red lines touch...,2,"{""gt"": 2, ""linewidth"": 2, ""left"": 2, ""resoluti..."


In [4]:
import os
os.path.join(os.getcwd(), 'images/abc.png')

'/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/abc.png'

In [5]:
df['image'] = df['image'].apply(lambda name: os.path.join(os.getcwd(), name))
df['answer'] = df['groundtruth']
df.head()

,task,image,prompt,groundtruth,metadata,answer
960,Line Plot Intersections,/mnt/disk1/luanborges/verifiers_vlm/verifiers/...,How many times do the blue and red lines touch...,2,"{""gt"": 2, ""linewidth"": 2, ""left"": 2, ""resoluti...",2
961,Line Plot Intersections,/mnt/disk1/luanborges/verifiers_vlm/verifiers/...,Count the intersection points where the blue a...,2,"{""gt"": 2, ""linewidth"": 2, ""left"": 2, ""resoluti...",2
962,Line Plot Intersections,/mnt/disk1/luanborges/verifiers_vlm/verifiers/...,How many times do the blue and red lines touch...,2,"{""gt"": 2, ""linewidth"": 4, ""left"": 4, ""resoluti...",2
963,Line Plot Intersections,/mnt/disk1/luanborges/verifiers_vlm/verifiers/...,Count the intersection points where the blue a...,2,"{""gt"": 2, ""linewidth"": 4, ""left"": 4, ""resoluti...",2
964,Line Plot Intersections,/mnt/disk1/luanborges/verifiers_vlm/verifiers/...,How many times do the blue and red lines touch...,2,"{""gt"": 2, ""linewidth"": 2, ""left"": 2, ""resoluti...",2


In [ ]:
df['prompt'] = """Your objective is to count how many times the blue and red lines touch each other
                  - The expected output is the correct order enclosed by <answer> </answer> tags.
                  - Use tags <think> </think> to think before answering."""

In [6]:
train_df = Dataset.from_pandas(df)

In [7]:
import verifiers as vf

print(vf.__version__)

model_name = "Qwen/Qwen2-VL-2B-Instruct"
model, tokenizer = vf.get_model_and_tokenizer(model_name)

vf_env = vf.MultimodalEnv(
    dataset=train_df)
dataset = vf_env.get_dataset()
eval_dataset = vf_env.get_eval_dataset(n=100)
rubric = vf_env.get_rubric()

# notable defaults: lr = 1e-6, max_grad_norm = 0.01, constant lr 10 warmup steps, 1024 tokens in+out
run_name = "multimodal_" + model_name.split("/")[-1].lower()
training_args = vf.get_default_grpo_config(
    run_name=run_name
)
trainer = vf.GRPOEnvTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=rubric,
    env=vf_env,
    args=training_args,
    train_dataset=dataset,
    #eval_dataset=eval_dataset,
)


INFO 04-10 11:13:59 [__init__.py:256] Automatically detected platform cuda.


2025-04-10 11:14:00,122	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


{'__module__': 'verifiers.server.vllm_client', '__doc__': '\n    A client class to interact with a vLLM server.\n\n    This class provides methods to generate completions, initialize and manage weight update groups, and update model\n    weights in a distributed setting. Before using it, start the vLLM server with `trl vllm-serve`.\n\n    Args:\n        host (`str`, *optional*, defaults to `"0.0.0.0"`):\n            IP address of the vLLM server.\n        server_port (`int`, *optional*, defaults to `8000`):\n            Port number of the vLLM server.\n        group_port (`int`, *optional*, defaults to `51216`):\n            Port number for the weight update group.\n        connection_timeout (`float`, *optional*, defaults to `0.0`):\n            Total timeout duration in seconds to wait for the server to be up. If the server is not up after the\n            timeout, a `ConnectionError` is raised.\n\n    Examples:\n        Run the vLLM server with the model `Qwen/Qwen2.5-7B`:\n\n      

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46
Map:  19%|█▉        | 693/3600 [00:00<00:00, 6867.33 examples/s]

Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.'}, {'type': 'image_url', 'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_960.png'}}]}]
Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'Count the intersection points where the blue and red lines meet. Put your answer in curly brackets, e.g., {2}.'}, {'type': 'image_url', 'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_961.png'}}]}]
Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.'}, {'type': 'image_url', 'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_962.png'}}]}]
Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'Count the intersection points wh

...

Map: 100%|██████████| 3600/3600 [00:00<00:00, 5865.27 examples/s]

Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.'}, {'type': 'image_url', 'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3520.png'}}]}]
Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'Count the intersection points where the blue and red lines meet. Put your answer in curly brackets, e.g., {2}.'}, {'type': 'image_url', 'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3521.png'}}]}]
Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.'}, {'type': 'image_url', 'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3522.png'}}]}]
Messages:  [{'role': 'user', 'content': [{'type': 'text', 'text': 'Count the intersection points

Dataset Promp:  [[{'content': [{'image_url': None, 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.', 'type': 'text'}, {'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_960.png'}, 'text': None, 'type': 'image_url'}], 'role': 'user'}], [{'content': [{'image_url': None, 'text': 'Count the intersection points where the blue and red lines meet. Put your answer in curly brackets, e.g., {2}.', 'type': 'text'}, {'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_961.png'}, 'text': None, 'type': 'image_url'}], 'role': 'user'}], [{'content': [{'image_url': None, 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.', 'type': 'text'}, {'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_962.png'}, 'text': None, 'type': 'image_url'}], 'role': 'user'}], [{'content': 

2025-04-10 11:14:13 - verifiers.server.vllm_client - INFO - Server is up!


INFO 04-10 11:14:13 [utils.py:925] Found nccl from library libnccl.so.2
INFO 04-10 11:14:13 [pynccl.py:69] vLLM is using nccl==2.21.5


In [8]:
trainer.train()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


wandb: Currently logged in as: luanborgesteodoro (franciscokunumi-universidade-federal-de-minas-gerais) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Inputs:  [{'task': 'Line Plot Intersections', 'image': '/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3702.png', 'prompt': [{'content': [{'image_url': None, 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.', 'type': 'text'}, {'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3702.png'}, 'text': None, 'type': 'image_url'}], 'role': 'user'}], 'groundtruth': '2', 'metadata': '{"gt": 2, "linewidth": 4, "left": 4, "resolution": 200, "distances": [3.0, 1.0, 2.0], "grid_size": 12}', 'answer': '2', '__index_level_0__': 3702}, {'task': 'Line Plot Intersections', 'image': '/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3702.png', 'prompt': [{'content': [{'image_url': None, 'text': 'How many times do the blue and red lines touch each other? Answer with a number in curly brackets, e.g., {5}.', 'type': 'text'}, {'image_url': {'url': 'file:/mnt/disk1/luanborges/verifiers_vlm/v

╭──────────────────────────────────────────────────── Step 0 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┓ │
│ ┃ Prompt                                                                            ┃ Completion     ┃ Reward ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━┩ │
│ │ [{'image_url': None, 'text': 'How many times do the blue and red lines touch each │ assistant: {3} │   0.04 │ │
│ │ other? Answer with a number in curly brackets, e.g., {5}.', 'type': 'text'},      │                │        │ │
│ │ {'image_url': {'url':                                                             │                │        │ │
│ │ 'file:/mnt/disk1/luanborges/verifiers_vlm/verifiers/images/img_3702.png'},        │                │        │ │
│ │ 'text': None, 'type': 'image_url'}]                                               │                │        │ │
│ └───────────────────────────────────────────────────────────────────────────────────┴────────────────┴────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 23.69 GiB of which 29.44 MiB is free. Process 241894 has 652.00 MiB memory in use. Including non-PyTorch memory, this process has 22.94 GiB memory in use. Of the allocated memory 20.97 GiB is allocated by PyTorch, and 1.56 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
trainer.vllm_client.close_communicator() 